# AusGrid2013 DP-FedAvg sweep: sampling rate 0.1

This notebook runs only sampling rate **0.1**.
It sweeps 4 noise multipliers × 3 clipping norms, with 20 rounds
per combination, using the deduplicated AusGrid2013 input.
There is no outer loop over sampling rates.

All three notebooks share `DP_fhf/outputs/ausgrid2013_sampling_sweep/`.
Full participation is stored under `full_participation/`;
partial runs use `amplified_v1/sampling_runs/q_<rate>__seed_fresh/`.
Each rate has its own `sweep_summary.csv`. After training, this
notebook also refreshes the parent's `all_sampling_rates_summary.csv`
from whichever rates have completed. Run the three notebooks one
at a time on a single GPU.

Summaries evaluate base forecasts. Existing reconciliation notebooks
for the older sweep directory do not automatically read this new
experiment. If a non-DP reference is available, its sampling rate is
1.0; it is not a matched partial-participation baseline.

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys

PROJECT_ROOT = os.environ.get("DP_FHF_PROJECT_ROOT")

def locate_project_root():
    candidates = [Path(PROJECT_ROOT).expanduser()] if PROJECT_ROOT else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "PPHF_modules" / "fhf").is_dir() and (candidate / "DP_fhf").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate the FHF project root. Set DP_FHF_PROJECT_ROOT or PROJECT_ROOT."
    )

ROOT = locate_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from PPHF_modules.dp_fhf.notebook_runtime import (
    locate_dataset_csv,
    run_covariance_notebook,
    run_fixed_dpfedavg,
    run_o2mr_notebook,
    run_sweep_dpfedavg,
    summarize_dpfedavg,
)

DRY_RUN = os.environ.get("DP_FHF_NOTEBOOK_DRY_RUN", "0") == "1"
print("Project root:", ROOT)
print("Python:", sys.executable)
print("Dry run:", DRY_RUN)

from PPHF_modules.dp_fhf.ausgrid_sampling_sweep import (
    run_ausgrid_sampling_sweep,
    summarize_ausgrid_sampling_sweeps,
)

In [ ]:
DATASET = "AusGrid2013"
SAMPLING_RATE = 0.1
SAMPLING_SEED = None
NUM_ROUNDS = 20
NOISE_MULTIPLIERS = [1.2, 0.9, 0.6, 0.3]
CLIP_NORMS = [1.0, 0.5, 0.2]
OVERWRITE = False
# DRY_RUN = True  # Uncomment to preview commands without training.

CSV_PATH = locate_dataset_csv(ROOT, DATASET)
OUTPUT_ROOT = ROOT / "DP_fhf" / "outputs" / "ausgrid2013_sampling_sweep"
print("Input:", CSV_PATH)
print("Sampling rate:", SAMPLING_RATE)
print("Runs:", len(NOISE_MULTIPLIERS) * len(CLIP_NORMS))
print("Common output folder:", OUTPUT_ROOT)

In [ ]:
summary = run_ausgrid_sampling_sweep(
    root=ROOT,
    csv_path=CSV_PATH,
    sampling_rate=SAMPLING_RATE,
    noise_multipliers=NOISE_MULTIPLIERS,
    clip_norms=CLIP_NORMS,
    num_rounds=NUM_ROUNDS,
    sampling_seed=SAMPLING_SEED,
    output_root=OUTPUT_ROOT,
    dry_run=DRY_RUN,
    overwrite=OVERWRITE,
    stop_on_error=False,
)
summary

In [ ]:
from PPHF_modules.dp_fhf.recover_dp_sweep import recover_ausgrid_sampling_sweep

recovered = recover_ausgrid_sampling_sweep(
    output_root=OUTPUT_ROOT,
    sampling_rate=SAMPLING_RATE,
    sampling_seed=SAMPLING_SEED,
    num_rounds=20,
)

recovered

In [ ]:
all_rates_summary = summarize_ausgrid_sampling_sweeps(
    OUTPUT_ROOT, dry_run=DRY_RUN,
)
all_rates_summary